# 第 7 章 · `nn.Module`：一棵自动登记参数的积木树

本章回答：写模型为什么要继承 `nn.Module`？`model.parameters()` 怎么知道有哪些参数？`state_dict` 是什么？`train()` / `eval()` 到底切换了什么？hook 能干嘛？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/07-nn-module.md`（讲解、动画、练习都在讲义里）。对应原文：[00-overview.md](../00-overview.md) §3；官方教程「Build the Neural Network」与 `torch.nn.Module` 文档（〔在线〕）；《深入浅出 PyTorch》第 5 章 模型定义（〔在线〕datawhalechina.github.io/thorough-pytorch）；本章输出在本机 torch 2.15.0 上跑过（代码见本章末尾"代码对照"）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 自动登记 | 40 + 27 | `count(MLP())` | 67 |
| 第一步 普通 list 不登记 | — | `count(PlainList())` | 只剩 head 的 34 个 |
| 第二步 模块树 | — | `named_modules()` | ['', 'fc1', 'drop', 'fc2'] |
| 第三步 Dropout | 留下的 × 1/(1−p) | `dropout_once()`（种子 0） | [0, 0, 2, 0, 0, 0, 2, 2] |
| 第三步 train / eval | — | `train_eval_repeat()` | (False, True) |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**类 `MLP`**（来自 `ch07_nn_module.py`）：下面要用到的小工具。

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.drop = nn.Dropout(0.5)
        self.fc2 = nn.Linear(8, 3)

    def forward(self, x):
        return self.fc2(self.drop(F.relu(self.fc1(x))))

**函数 `count`**（来自 `ch07_nn_module.py`）：下面要用到的小工具。

In [ ]:
def count(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

**函数 `dropout_once`**（来自 `ch07_nn_module.py`）：下面要用到的小工具。

In [ ]:
def dropout_once(seed: int = 0):
    torch.manual_seed(seed)
    d = nn.Dropout(0.5)
    d.train()
    return d(torch.ones(8)).tolist()

**函数 `train_eval_repeat`**（来自 `ch07_nn_module.py`）：train 时两次输出不同、eval 时相同。

In [ ]:
def train_eval_repeat():
    """train 时两次输出不同、eval 时相同。"""
    torch.manual_seed(0)
    m = MLP()
    x = torch.ones(1, 4)
    m.train()
    a, b = m(x), m(x)
    m.eval()
    c, d = m(x), m(x)
    return torch.equal(a, b), torch.equal(c, d)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
m = MLP()

In [ ]:
print("\n第一步 named_parameters：", [(n, tuple(p.shape)) for n, p in m.named_parameters()])
print("参数总数：", count(m))

In [ ]:
print("\n第二步 模块树：", [n for n, _ in m.named_modules()])
print("state_dict 键：", list(m.state_dict().keys()))

In [ ]:
print("\n第三步 Dropout 训练时：", dropout_once())
print("train 两次相同？eval 两次相同？", train_eval_repeat())

In [ ]:
print("\n第四步 hook")
h = m.fc1.register_forward_hook(lambda mod, inp, out: print("  fc1 输出形状", tuple(out.shape)))
m(torch.ones(1, 4))
h.remove()

## 3. 代码里才看得到的

1. **`nn.Linear(4, 8).weight` 是 (8, 4)**：存的是转置，输出维在前。
2. **普通 list 不报错**：参数只是悄悄没进 `parameters()`，训练时那几层不更新。

## 4. 自己改一改

- 把 `PlainList` 的 list 换成 `nn.ModuleList`，再数参数。
- 换几个随机种子调 `dropout_once`，看每次丢的位置不同、但留下的都是 2。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 继承 Module 有 HR，赋给 self 自动登记；普通 list 不登记，要用 ModuleList；
> 模块成树、路径成名，state_dict 是名字到张量的清单；
> train/eval 切层的行为，不管梯度；hook 是进出口的探头。